In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import re
import shutil
import numpy as np
import pandas as pd
import torch
import evaluate
import nltk
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [ ]:
MODEL_NAME       = "t5-base"
MAX_INPUT_LEN    = 512       
MAX_TARGET_LEN   = 256
BATCH_SIZE       = 4
GRAD_ACCUM_STEPS = 4         
NUM_EPOCHS       = 3
LR               = 3e-5
OUTPUT_DIR       = "./t5_mixed_output"   

# ── Google Drive paths ────────────────────────────────────────────────────────
DATASET_PATH    = "/content/mixed_meeting_dataset (1).csv"  
DRIVE_SAVE_PATH = "/content/drive/MyDrive/t5_mixed_best_model"

In [ ]:
print("Loading pre-mixed dataset...")
df_mixed = pd.read_csv(DATASET_PATH)

print(f"Total Mixed Dataset Size: {len(df_mixed)}")
print(df_mixed.head())

df_clean = df_mixed[["Transcript", "Summary"]].dropna().reset_index(drop=True)

full_dataset = Dataset.from_pandas(df_clean)
split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset  = split_dataset["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")

Loading pre-mixed dataset...
Total Mixed Dataset Size: 1460
                                          Transcript  \
0  Speaker 3: increases applicable. To certain re...   
1  Speaker 0: Okay. Our last agenda item is the m...   
2  #Person1#: I hear you are moving to Dalian. #P...   
3  #Person1#: Would you like to see our new shirt...   
4  Speaker 1: Please read item 31. Speaker 7: The...   

                                             Summary  
0  Adoption of Resolution Calling an Election to ...  
1  A MOTION requesting the executive establish an...  
2  #Person2#'s moving to Dalian for work. #Person...  
3  #Person1# tries to sell the new shirts to #Per...  
4  AN ORDINANCE authorizing the Director of Seatt...  
Train size: 1168, Eval size: 292


In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

def clean_text(text):
    text = re.sub(r'#(Person\d+)#:', r'\1:', text)
    text = re.sub(r'#(Person\d+)#',  r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def preprocess_function(examples):
    inputs  = ["summarize: " + clean_text(t) for t in examples["Transcript"]]
    targets = [clean_text(s)                 for s in examples["Summary"]]
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing training dataset...")
tokenized_train = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)
print("Tokenizing eval dataset...")
tokenized_eval = eval_dataset.map(
    preprocess_function, batched=True, remove_columns=eval_dataset.column_names
)
print(f"Train tokens: {len(tokenized_train)}, Eval tokens: {len(tokenized_eval)}")


In [ ]:
nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Replace out-of-range values, then convert to plain Python ints
    # (.tolist() is required — T5's Rust tokenizer crashes on numpy.int64 in Python 3.12)
    predictions    = np.where(predictions < 0, tokenizer.pad_token_id, predictions)
    decoded_preds  = tokenizer.batch_decode(predictions.tolist(), skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels.tolist(), skip_special_tokens=True)
    # ROUGE expects a newline after each sentence
    decoded_preds  = ["\n".join(nltk.sent_tokenize(pred.strip()))  for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # Add mean generated length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    return {k: round(v, 4) for k, v in result.items()}

In [ ]:
model    = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    warmup_ratio=0.1,
    label_smoothing_factor=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=4,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainable parameters: 222,903,552


In [ ]:
print("Starting fine-tuning T5 on mixed dataset...")
train_result = trainer.train()
print("\nTraining complete.")
print(train_result.metrics)

Starting fine-tuning T5 on mixed dataset...


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,25.135920,3.888093,0.397700,0.240900,0.349500,0.363100,91.657500
2,16.636730,3.674573,0.431000,0.274000,0.383800,0.401200,87.219200
3,15.438099,3.651121,0.438900,0.278700,0.390200,0.409400,86.589000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training complete.
{'train_runtime': 1130.8574, 'train_samples_per_second': 3.099, 'train_steps_per_second': 0.194, 'total_flos': 2082336264253440.0, 'train_loss': 18.04516977806614, 'epoch': 3.0}


In [ ]:
print("Running evaluation on eval set...")
eval_results = trainer.evaluate()

print("\n=== EVALUATION RESULTS ===")
for key, value in eval_results.items():
    print(f"  {key}: {value}")

Running evaluation on eval set...



=== EVALUATION RESULTS ===
  eval_loss: 3.6511213779449463
  eval_rouge1: 0.4389
  eval_rouge2: 0.2787
  eval_rougeL: 0.3902
  eval_rougeLsum: 0.4094
  eval_gen_len: 86.589
  eval_runtime: 234.567
  eval_samples_per_second: 1.245
  eval_steps_per_second: 0.311
  epoch: 3.0


In [ ]:
N = 5
sample_ds  = eval_dataset.select(range(N))
sample_tok = sample_ds.map(preprocess_function, batched=True, remove_columns=sample_ds.column_names)

pred_out   = trainer.predict(sample_tok)
pred_ids   = np.where(pred_out.predictions < 0, tokenizer.pad_token_id, pred_out.predictions)
decoded    = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

for i in range(N):
    print(f"\n[Sample {i+1}]")
    print(f"Transcript : {clean_text(eval_dataset[i]['Transcript'])[:300]}...")
    print(f"Reference  : {clean_text(eval_dataset[i]['Summary'])}")
    print(f"Predicted  : {decoded[i]}")
    print("-" * 70)


In [ ]:
print("Saving best model to Google Drive...")

# Remove previous save if it exists
if os.path.exists(DRIVE_SAVE_PATH):
    shutil.rmtree(DRIVE_SAVE_PATH)

trainer.save_model(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

print(f"Best model saved to: {DRIVE_SAVE_PATH}")
print("Files saved:")
for f in os.listdir(DRIVE_SAVE_PATH):
    size = os.path.getsize(os.path.join(DRIVE_SAVE_PATH, f)) / 1e6
    print(f"  {f}  ({size:.1f} MB)")

Saving best model to Google Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to: /content/drive/MyDrive/t5_mixed_best_model
Files saved:
  config.json  (0.0 MB)
  generation_config.json  (0.0 MB)
  model.safetensors  (891.6 MB)
  tokenizer_config.json  (0.0 MB)
  tokenizer.json  (2.1 MB)
  training_args.bin  (0.0 MB)
